# CLM-0.3b — Marginal Growth Utility

Formal saturated-growth experiment. The growth operator is frozen from CLM-0.3; this notebook tests marginal capacity utility and WHERE selection.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
REPO_URL = 'https://github.com/ArcheLabs/mini-cells.git'
BRANCH = 'codex/clm-0.3b-marginal-growth-utility'

if not (ROOT / '.git').exists():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(f'{ROOT} exists but is not a git checkout')
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT, check=True)

CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
CODE_TREE = subprocess.check_output(['git', 'rev-parse', 'HEAD^{tree}'], cwd=ROOT, text=True).strip()
print('branch:', BRANCH)
print('commit:', CODE_COMMIT)
print('tree:', CODE_TREE)
sys.path.insert(0, str(ROOT / 'research'))

In [ ]:
import torch
print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
})
if not torch.cuda.is_available():
    raise RuntimeError('CLM-0.3b formal execution requires CUDA')
release = ROOT / 'artifacts/releases/clm-0.1/model.pt'
observed = hashlib.sha256(release.read_bytes()).hexdigest()
assert observed == '87d36c408ae3873ffd567ebf17050661b42ddae2c8d5d1bab84b2c27c3c7e7a0'
print('CLM-0.1 SHA-256:', observed)

## Mandatory preflight

Do not start the formal matrix unless all tests pass. Any code change after a checkpoint invalidates resume by design.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_clm_marginal_growth.py',
    'tests/test_clm_progressive_growth.py',
    'tests/test_growth_router.py',
    'tests/test_growth_checkpoint.py',
    '-q',
], cwd=ROOT, check=True)

In [ ]:
# Plan-only by default.
RESULTS = ROOT / 'results/clm-0.3b-marginal-growth-utility'
runner = [sys.executable, 'scripts/run_clm_marginal_growth_002.py', '--output-root', str(RESULTS)]
subprocess.run(runner, cwd=ROOT, check=True)

## Formal 3×3 matrix

Set `RUN_FORMAL=True` only after reviewing the plan. With two T4 GPUs the parent runs two workers concurrently and auto-resumes only checkpoints produced by the exact same code commit.

In [ ]:
RUN_FORMAL = False
if RUN_FORMAL:
    subprocess.run([*runner, '--execute'], cwd=ROOT, check=True)
else:
    print('FORMAL RUN DISABLED')

In [ ]:
decision_path = RESULTS / 'decision.json'
if decision_path.exists():
    decision = json.loads(decision_path.read_text(encoding='utf-8'))
    print(json.dumps(decision, indent=2, sort_keys=True))
    print('formal history:', RESULTS / 'formal-ppl-history.csv')
    print('replicate summary:', RESULTS / 'replicate-summary.json')
else:
    print('NO FORMAL DECISION YET')

## Publish curated evidence

Publication verifies that all nine workers used the same immutable training commit/tree before creating a result branch.

In [ ]:
PUBLISH = False
if PUBLISH:
    subprocess.run([
        sys.executable, 'scripts/publish_clm_0_3b_results.py', '--push'
    ], cwd=ROOT, check=True)
else:
    print('PUBLICATION DISABLED')